### **Primality Tests: Trial Division, Fermat, and Miller-Rabin**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Primality_Tests.ipynb)

> Key generation for RSA, Diffie-Hellman, and ElGamal needs large random primes. **Trial division** proves primality but is far too slow for big numbers. The **Fermat test** is fast but can be fooled by **Carmichael numbers**. **Miller-Rabin** fixes that and is what real libraries use.

In [1]:
# Trial division: exact, but needs up to sqrt(n) / 2 divisions
import math
import secrets
import time

def is_prime_trial(n):
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    return all(n % d for d in range(3, math.isqrt(n) + 1, 2))

for n in (2, 9, 97, 561, 1_000_000_007):
    print(f"{n:>13}: {'prime' if is_prime_trial(n) else 'composite'}")

start = time.perf_counter()
is_prime_trial(999_999_999_989)                       # a 40-bit prime
print(f"40-bit prime by trial division: {time.perf_counter() - start:.2f} s (a 1024-bit prime would never finish)")

            2: prime
            9: composite
           97: prime
          561: composite
   1000000007: prime
40-bit prime by trial division: 0.02 s (a 1024-bit prime would never finish)


#### **01. Fermat Test and Carmichael Numbers**

Fermat's little theorem: if `p` is prime, `a^(p−1) ≡ 1 (mod p)` for every `a` not divisible by `p`. The test checks this for random bases. **Carmichael numbers** such as 561 = 3 · 11 · 17 satisfy it for **every** base coprime to them, so the test says "prime" for a composite number.

In [2]:
# Fermat test (bases from the secrets module) and its weakness
def fermat_test(n, rounds=20):
    if n < 4:
        return n in (2, 3)
    for _ in range(rounds):
        a = secrets.randbelow(n - 3) + 2              # base in [2, n-2]
        if pow(a, n - 1, n) != 1:
            return False
    return True

carmichael = 561
liars = sum(1 for a in range(2, carmichael - 1) if pow(a, carmichael - 1, carmichael) == 1)
print(f"561 = 3 x 11 x 17 | Fermat liars: {liars} of {carmichael - 3} bases ({liars / (carmichael - 3):.0%})")
print("Fermat test on 2^127-1 (prime):", fermat_test(2 ** 127 - 1))
print("Fermat test on 341 (composite, 20 rounds):", fermat_test(341))

561 = 3 x 11 x 17 | Fermat liars: 318 of 558 bases (57%)
Fermat test on 2^127-1 (prime): True
Fermat test on 341 (composite, 20 rounds): False


#### **02. Miller-Rabin**

Write `n − 1 = 2^s · d`. For a prime, the sequence `a^d, a^(2d), …, a^(2^s·d)` must reach 1 only through `n − 1`. A composite number passes one round with probability at most 1/4, so 40 rounds give an error below 2^−80, and **no composite number fools it for every base**.

In [3]:
# Miller-Rabin catches the Carmichael numbers that fool Fermat
def miller_rabin(n, rounds=40):
    if n < 4:
        return n in (2, 3)
    if n % 2 == 0:
        return False
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    for _ in range(rounds):
        x = pow(secrets.randbelow(n - 3) + 2, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False                              # a is a witness: n is composite
    return True

carmichael_numbers = [561, 1105, 1729, 2465, 2821, 6601, 8911]
print(f"{'n':>6} | {'Fermat base 2':>13} | {'Miller-Rabin':>12} | trial division")
for n in carmichael_numbers:
    print(f"{n:>6} | {str(pow(2, n - 1, n) == 1):>13} | {str(miller_rabin(n)):>12} | {is_prime_trial(n)}")
    assert not miller_rabin(n)

# Agreement with trial division on every number below 20,000
assert all(miller_rabin(n) == is_prime_trial(n) for n in range(20_000))
print("\nMiller-Rabin agrees with trial division for every n < 20,000")

     n | Fermat base 2 | Miller-Rabin | trial division
   561 |          True |        False | False
  1105 |          True |        False | False
  1729 |          True |        False | False
  2465 |          True |        False | False
  2821 |          True |        False | False
  6601 |          True |        False | False
  8911 |          True |        False | False

Miller-Rabin agrees with trial division for every n < 20,000


#### **03. Generating a Large Random Prime**

Key generation picks random odd numbers of the right size (top bit set) and tests them until one is prime. By the prime number theorem, about 1 in `ln(2^1024)/2 ≈ 355` odd 1024-bit numbers is prime.

In [4]:
# Generate a 1024-bit prime with Miller-Rabin and secure randomness
def random_prime(bits):
    attempts = 0
    while True:
        attempts += 1
        candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1
        if miller_rabin(candidate):
            return candidate, attempts

start = time.perf_counter()
prime, attempts = random_prime(1024)
print(f"1024-bit prime found after {attempts} candidates in {time.perf_counter() - start:.2f} s")
print(f"{str(prime)[:40]}... ({prime.bit_length()} bits)")

1024-bit prime found after 302 candidates in 1.21 s
1136870626122335056812285487722614184277... (1024 bits)


#### **Key Takeaways**

* Trial division is exact but exponential in the number of bits.
* The Fermat test is fooled by Carmichael numbers; never use it alone.
* Miller-Rabin with enough rounds and **secure random bases** is the standard probabilistic test.